In [7]:
%pip install numpy scipy

  Using cached scipy-1.16.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
Using cached scipy-1.16.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (35.9 MB)
Note: you may need to restart the kernel to use updated packages.


# **Results: AAD through the solver**
## Crank–Nicolson FD

In [1]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm
import time
from aad.core.var import ADVar
from aad.core.engine import reverse, zero_adjoints

# ========== BSM Closed-form ==========
def bsm_price(S0, K, T, r, sigma, cp_flag='C'):
    d1 = (log(S0/K) + (r+0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    if cp_flag == 'C':
        return S0*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)
    else:
        return K*exp(-r*T)*norm.cdf(-d2) - S0*norm.cdf(-d1)

def bsm_vega_rho(S0, K, T, r, sigma, cp_flag='C'):
    d1 = (log(S0/K) + (r+0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    vega = S0*norm.pdf(d1)*sqrt(T)
    if cp_flag == 'C':
        rho = K*T*exp(-r*T)*norm.cdf(d2)
    else:
        rho = -K*T*exp(-r*T)*norm.cdf(-d2)
    return vega, rho

# ========== PDE CN Solver ==========
def tri_coeffs_bs(M, r, sigma):
    i = list(range(1, M))
    A = [0.5*sigma**2*j**2 - 0.5*r*j for j in i]
    B = [-sigma**2*j**2 - r for j in i]
    C = [0.5*sigma**2*j**2 + 0.5*r*j for j in i]
    return A, B, C

def thomas_algorithm(lower, main, upper, rhs):
    n = len(rhs)
    a = lower[:]
    b = main[:]
    c = upper[:]
    d = rhs[:]

    for i in range(1, n):
        m = a[i-1] / b[i-1]
        b[i] = b[i] - m * c[i-1]
        d[i] = d[i] - m * d[i-1]

    x = [None]*n
    x[-1] = d[-1]/b[-1]
    for i in range(n-2, -1, -1):
        x[i] = (d[i] - c[i]*x[i+1]) / b[i]
    return x

def pde_cn_price(S0, K, T, r, sigma, cp_flag='C', M=200, N=200):
    Smax = 4*K
    dS = Smax/M
    dt = T/N
    S = [j*dS for j in range(M+1)]

    if cp_flag == 'C':
        V = [max(s-K,0) for s in S]
    else:
        V = [max(K-s,0) for s in S]

    for _ in range(N):
        A,B,C = tri_coeffs_bs(M, r, sigma)
        lower = [-0.5*dt*a for a in A[1:]]
        main  = [1 - 0.5*dt*b for b in B]
        upper = [-0.5*dt*c for c in C[:-1]]

        rhs = []
        for j in range(1,M):
            rhs_j = (1+0.5*dt*B[j-1])*V[j] \
                    + 0.5*dt*(A[j-1]*V[j-1] + C[j-1]*V[j+1])
            rhs.append(rhs_j)

        sol = thomas_algorithm(lower, main, upper, rhs)
        for j in range(1,M):
            V[j] = sol[j-1]

    j = int(S0/dS)
    w = (S0 - S[j])/dS
    return (1-w)*V[j] + w*V[j+1]

# ========== AD Greeks ==========
def ad_greeks(S0,K,T,r,sigma,cp_flag):
    zero_adjoints()
    r_ad     = ADVar(r, name="r")
    sigma_ad = ADVar(sigma, name="sigma")
    price    = pde_cn_price(S0,K,T,r_ad,sigma_ad,cp_flag)
    reverse(price)
    return price.val, sigma_ad.adj, r_ad.adj

# ========== Demo ==========
S0,K,T,r,sigma = 100,100,1.0,0.05,0.2

# 1. BSM Price
print("=== BSM Price ===")
t0=time.perf_counter()
p_bsm = bsm_price(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
print(f"Price={p_bsm:.6f} (time {t1-t0:.4f}s)")

# 2. PDE Price
print("\n=== PDE Price (CN scheme) ===")
t0=time.perf_counter()
p_pde = pde_cn_price(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
print(f"Price={p_pde:.6f} (time {t1-t0:.4f}s)")

# 3. BSM Greeks
print("\n=== BSM Greeks ===")
t0=time.perf_counter()
v_bsm,r_bsm = bsm_vega_rho(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
print(f"Vega={v_bsm:.6f}, Rho={r_bsm:.6f} (time {t1-t0:.4f}s)")

# 4. FD Greeks
print("\n=== FD Greeks (bumping) ===")
t0=time.perf_counter()
h=1e-4
p_fd = pde_cn_price(S0,K,T,r,sigma,'C')
v_fd = (pde_cn_price(S0,K,T,r,sigma+h,'C')-p_fd)/h
r_fd = (pde_cn_price(S0,K,T,r+h,sigma,'C')-p_fd)/h
t1=time.perf_counter()
err_v = abs(v_fd-v_bsm)/abs(v_bsm)
err_r = abs(r_fd-r_bsm)/abs(r_bsm)
print(f"Vega={v_fd:.6f}, Rho={r_fd:.6f} (time {t1-t0:.4f}s)")
print(f"RelErr Vega={err_v:.2%}, Rho={err_r:.2%}")

# 5. AD Greeks
print("\n=== AD Greeks (reverse-mode) ===")
t0=time.perf_counter()
p_ad,v_ad,r_ad = ad_greeks(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
err_v = abs(v_ad-v_bsm)/abs(v_bsm)
err_r = abs(r_ad-r_bsm)/abs(r_bsm)
print(f"Vega={v_ad:.6f}, Rho={r_ad:.6f} (time {t1-t0:.4f}s)")
print(f"RelErr Vega={err_v:.2%}, Rho={err_r:.2%}")

=== BSM Price ===
Price=10.450584 (time 0.0005s)

=== PDE Price (CN scheme) ===
Price=10.440692 (time 0.0677s)

=== BSM Greeks ===
Vega=37.524035, Rho=53.232482 (time 0.0005s)

=== FD Greeks (bumping) ===
Vega=37.578452, Rho=53.232693 (time 0.1635s)
RelErr Vega=0.15%, Rho=0.00%

=== AD Greeks (reverse-mode) ===
Vega=37.577988, Rho=53.225983 (time 23.2343s)
RelErr Vega=0.14%, Rho=0.01%


## Explicit FD

In [2]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm
import time
from aad.core.var import ADVar
from aad.core.engine import reverse, zero_adjoints

# ========== BSM Closed-form ==========
def bsm_price(S0, K, T, r, sigma, cp_flag='C'):
    d1 = (log(S0/K) + (r+0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    if cp_flag == 'C':
        return S0*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)
    else:
        return K*exp(-r*T)*norm.cdf(-d2) - S0*norm.cdf(-d1)

def bsm_vega_rho(S0, K, T, r, sigma, cp_flag='C'):
    d1 = (log(S0/K) + (r+0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    vega = S0*norm.pdf(d1)*sqrt(T)
    if cp_flag == 'C':
        rho = K*T*exp(-r*T)*norm.cdf(d2)
    else:
        rho = -K*T*exp(-r*T)*norm.cdf(-d2)
    return vega, rho

def pde_explicit_price(S0, K, T, r, sigma, cp_flag='C', M=200, safety=0.9):
    # 如果 r/sigma 是 ADVar，取它们的 val 来算 CFL
    r_val     = r.val if isinstance(r, ADVar) else r
    sigma_val = sigma.val if isinstance(sigma, ADVar) else sigma
    
    Smax = 4*K
    dS = Smax/M
    
    # CFL 稳定条件： dt <= 1 / (σ² M² + r M)
    dt_max = 1.0 / (sigma_val**2 * M**2 + r_val*M)
    N = int(np.ceil(T / (safety * dt_max)))
    dt = T / N
    
    # 初始化 payoff —— 保持与参数同类型
    def to_same_type(x):
        if isinstance(r, ADVar) or isinstance(sigma, ADVar):
            return ADVar(float(x))   # 包装成 ADVar
        else:
            return float(x)
    
    S = np.linspace(0, Smax, M+1)
    if cp_flag == 'C':
        V = [to_same_type(max(s-K,0)) for s in S]
    else:
        V = [to_same_type(max(K-s,0)) for s in S]
    
    # 显式迭代
    for _ in range(N):
        V_new = V.copy()
        for j in range(1,M):
            a = 0.5*dt*(sigma**2*j**2 - r*j)
            b = 1 - dt*(sigma**2*j**2 + r)
            c = 0.5*dt*(sigma**2*j**2 + r*j)
            V_new[j] = a*V[j-1] + b*V[j] + c*V[j+1]
        V = V_new
    
    # 线性插值
    j = int(S0/dS)
    w = (S0 - S[j])/dS
    return (1-w)*V[j] + w*V[j+1]

# ========== AD Greeks ==========
def ad_greeks(S0,K,T,r,sigma,cp_flag):
    zero_adjoints()
    r_ad     = ADVar(r, name="r")
    sigma_ad = ADVar(sigma, name="sigma")
    price    = pde_explicit_price(S0,K,T,r_ad,sigma_ad,cp_flag)
    reverse(price)
    return price.val, sigma_ad.adj, r_ad.adj

# ========== Demo ==========
S0,K,T,r,sigma = 100,100,1.0,0.05,0.2

# 1. BSM Price
print("=== BSM Price ===")
t0=time.perf_counter()
p_bsm = bsm_price(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
print(f"Price={p_bsm:.6f} (time {t1-t0:.4f}s)")

# 2. PDE Price (Explicit scheme)
print("\n=== PDE Price (Explicit scheme) ===")
t0=time.perf_counter()
p_pde = pde_explicit_price(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
print(f"Price={p_pde:.6f} (time {t1-t0:.4f}s)")

# 3. BSM Greeks
print("\n=== BSM Greeks ===")
t0=time.perf_counter()
v_bsm,r_bsm = bsm_vega_rho(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
print(f"Vega={v_bsm:.6f}, Rho={r_bsm:.6f} (time {t1-t0:.4f}s)")

# 4. FD Greeks (bumping)
print("\n=== FD Greeks (bumping) ===")
t0=time.perf_counter()
h=1e-4
p_fd = pde_explicit_price(S0,K,T,r,sigma,'C')
v_fd = (pde_explicit_price(S0,K,T,r,sigma+h,'C')-p_fd)/h
r_fd = (pde_explicit_price(S0,K,T,r+h,sigma,'C')-p_fd)/h
t1=time.perf_counter()
err_v = abs(v_fd-v_bsm)/abs(v_bsm)
err_r = abs(r_fd-r_bsm)/abs(r_bsm)
print(f"Vega={v_fd:.6f}, Rho={r_fd:.6f} (time {t1-t0:.4f}s)")
print(f"RelErr Vega={err_v:.2%}, Rho={err_r:.2%}")

# 5. AD Greeks (reverse-mode)
print("\n=== AD Greeks (reverse-mode) ===")
t0=time.perf_counter()
p_ad,v_ad,r_ad = ad_greeks(S0,K,T,r,sigma,'C')
t1=time.perf_counter()
err_v = abs(v_ad-v_bsm)/abs(v_bsm)
err_r = abs(r_ad-r_bsm)/abs(r_bsm)
print(f"Vega={v_ad:.6f}, Rho={r_ad:.6f} (time {t1-t0:.4f}s)")
print(f"RelErr Vega={err_v:.2%}, Rho={err_r:.2%}")

=== BSM Price ===
Price=10.450584 (time 0.0006s)

=== PDE Price (Explicit scheme) ===
Price=10.441274 (time 0.2391s)

=== BSM Greeks ===
Vega=37.524035, Rho=53.232482 (time 0.0004s)

=== FD Greeks (bumping) ===
Vega=37.575057, Rho=53.233055 (time 0.7222s)
RelErr Vega=0.14%, Rho=0.00%

=== AD Greeks (reverse-mode) ===
Vega=37.581170, Rho=53.226346 (time 124.0123s)
RelErr Vega=0.15%, Rho=0.01%


# **Results: Discrete adjoint AAD**

In [5]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm
import time

# ========== BSM Closed-form ==========
def bsm_price(S0, K, T, r, sigma, cp_flag='C'):
    d1 = (log(S0/K) + (r+0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    if cp_flag == 'C':
        return S0*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)
    else:
        return K*exp(-r*T)*norm.cdf(-d2) - S0*norm.cdf(-d1)

def bsm_vega_rho(S0, K, T, r, sigma, cp_flag='C'):
    d1 = (log(S0/K) + (r+0.5*sigma**2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    vega = S0*norm.pdf(d1)*sqrt(T)
    if cp_flag == 'C':
        rho = K*T*exp(-r*T)*norm.cdf(d2)
    else:
        rho = -K*T*exp(-r*T)*norm.cdf(-d2)
    return vega, rho

# ========== CN solver ==========
def crank_nicolson(S0,K,T,r,sigma,cp_flag='C',M=200,N=200):
    Smax = 4*K
    dS = Smax/M
    dt = T/N
    S = np.linspace(0,Smax,M+1)

    # payoff
    if cp_flag=='C':
        V = np.maximum(S-K,0.0)
    else:
        V = np.maximum(K-S,0.0)

    # 系数 (时间常数)
    j = np.arange(0,M+1)
    alpha = 0.25*dt*(sigma**2*j**2 - r*j)
    beta  = -0.5*dt*(sigma**2*j**2 + r)
    gamma = 0.25*dt*(sigma**2*j**2 + r*j)

    # 构造 A,B 矩阵的三对角系数
    a = -alpha[1:M]
    b = 1-beta[1:M]
    c = -gamma[1:M]
    A = np.diag(b) + np.diag(a[1:],-1) + np.diag(c[:-1],1)

    a = alpha[1:M]
    b = 1+beta[1:M]
    c = gamma[1:M]
    B = np.diag(b) + np.diag(a[1:],-1) + np.diag(c[:-1],1)

    # time stepping
    V_hist = []
    for n in range(N):
        rhs = B.dot(V[1:M])
        V[1:M] = np.linalg.solve(A, rhs)
        V[0] = 0
        V[M] = Smax-K*exp(-r*dt*(n+1)) if cp_flag=='C' else K*exp(-r*dt*(n+1))
        V_hist.append(V.copy())
    return V, V_hist, dS, dt, S

# ========== Discrete adjoint ==========
def adjoint_greeks(S0,K,T,r,sigma,cp_flag='C',M=200,N=200):
    V, V_hist, dS, dt, S = crank_nicolson(S0,K,T,r,sigma,cp_flag,M,N)

    # 插值
    j = int(S0/dS)
    w = (S0 - S[j])/dS
    e = np.zeros(M+1)
    e[j]   = 1-w
    e[j+1] = w

    # 初始化伴随变量
    lam = e.copy()

    grad_sigma = 0.0
    grad_r = 0.0

    # 回溯 (discrete adjoint)
    for n in reversed(range(N)):
        Vn1 = V_hist[n]
        Vn  = V_hist[n-1] if n>0 else V_hist[0]*0.0

        jidx = np.arange(0,M+1)
        alpha = 0.25*dt*(sigma**2*jidx**2 - r*jidx)
        beta  = -0.5*dt*(sigma**2*jidx**2 + r)
        gamma = 0.25*dt*(sigma**2*jidx**2 + r*jidx)

        # A, B 矩阵
        aA = -alpha[1:M]; bA = 1-beta[1:M]; cA = -gamma[1:M]
        A  = np.diag(bA)+np.diag(aA[1:],-1)+np.diag(cA[:-1],1)

        aB = alpha[1:M]; bB = 1+beta[1:M]; cB = gamma[1:M]
        B  = np.diag(bB)+np.diag(aB[1:],-1)+np.diag(cB[:-1],1)

        # 伴随系统: A^T lam^n = B^T lam^{n+1}
        rhs = B.T.dot(lam[1:M])
        lam_inner = np.linalg.solve(A.T,rhs)
        lam_new = np.zeros(M+1)
        lam_new[1:M] = lam_inner
        lam = lam_new

        # 累加梯度项 (对 σ 和 r 的偏导)
        d_alpha_sigma = 0.5*dt*(sigma*jidx**2)
        d_beta_sigma  = -dt*(sigma*jidx**2)
        d_gamma_sigma = 0.5*dt*(sigma*jidx**2)

        d_alpha_r = -0.25*dt*jidx
        d_beta_r  = -0.5*dt*np.ones_like(jidx)
        d_gamma_r = 0.25*dt*jidx

        # 构造 dA/dθ, dB/dθ 在内部点
        da_sigma = -d_alpha_sigma[1:M]; db_sigma = -d_beta_sigma[1:M]; dc_sigma = -d_gamma_sigma[1:M]
        da_r = -d_alpha_r[1:M]; db_r = -d_beta_r[1:M]; dc_r = -d_gamma_r[1:M]

        dA_sigma = np.diag(db_sigma)+np.diag(da_sigma[1:],-1)+np.diag(dc_sigma[:-1],1)
        dA_r     = np.diag(db_r)+np.diag(da_r[1:],-1)+np.diag(dc_r[:-1],1)

        daB_sigma = d_alpha_sigma[1:M]; dbB_sigma = d_beta_sigma[1:M]; dcB_sigma = d_gamma_sigma[1:M]
        daB_r = d_alpha_r[1:M]; dbB_r = d_beta_r[1:M]; dcB_r = d_gamma_r[1:M]

        dB_sigma = np.diag(dbB_sigma)+np.diag(daB_sigma[1:],-1)+np.diag(dcB_sigma[:-1],1)
        dB_r     = np.diag(dbB_r)+np.diag(daB_r[1:],-1)+np.diag(dcB_r[:-1],1)

        grad_sigma += lam[1:M] @ (dB_sigma.dot(Vn[1:M]) - dA_sigma.dot(Vn1[1:M]))
        grad_r     += lam[1:M] @ (dB_r.dot(Vn[1:M])     - dA_r.dot(Vn1[1:M]))

    price = e @ V
    return price, grad_sigma, grad_r

# ========== Demo ==========
if __name__ == "__main__":
    S0,K,T,r,sigma = 100,100,1.0,0.05,0.2

    # 1. BSM Price
    print("=== BSM Price ===")
    t0=time.perf_counter()
    p_bsm = bsm_price(S0,K,T,r,sigma,'C')
    t1=time.perf_counter()
    print(f"Price={p_bsm:.6f} (time {t1-t0:.4f}s)")

    # 2. PDE Price
    print("\n=== PDE Price (CN scheme) ===")
    t0=time.perf_counter()
    V,_,_,_,S = crank_nicolson(S0,K,T,r,sigma,'C')
    dS=S[1]-S[0]
    j=int(S0/dS)
    w=(S0-S[j])/dS
    p_pde=(1-w)*V[j]+w*V[j+1]
    t1=time.perf_counter()
    print(f"Price={p_pde:.6f} (time {t1-t0:.4f}s)")

    # 3. BSM Greeks
    print("\n=== BSM Greeks ===")
    t0=time.perf_counter()
    v_bsm,r_bsm = bsm_vega_rho(S0,K,T,r,sigma,'C')
    t1=time.perf_counter()
    print(f"Vega={v_bsm:.6f}, Rho={r_bsm:.6f} (time {t1-t0:.4f}s)")

    # 4. FD Greeks (bumping)
    print("\n=== FD Greeks (bumping) ===")
    t0=time.perf_counter()
    h=1e-4
    V0,_,_,_,S = crank_nicolson(S0,K,T,r,sigma,'C')
    p0=(1-w)*V0[j]+w*V0[j+1]
    Vp,_,_,_,S = crank_nicolson(S0,K,T,r,sigma+h,'C')
    pp=(1-w)*Vp[j]+w*Vp[j+1]
    v_fd=(pp-p0)/h
    Vp,_,_,_,S = crank_nicolson(S0,K,T,r+h,sigma,'C')
    pp=(1-w)*Vp[j]+w*Vp[j+1]
    r_fd=(pp-p0)/h
    t1=time.perf_counter()
    print(f"Vega={v_fd:.6f}, Rho={r_fd:.6f} (time {t1-t0:.4f}s)")

    # 5. Adjoint Greeks
    print("\n=== Adjoint Greeks (discrete) ===")
    t0=time.perf_counter()
    p_adj,v_adj,r_adj = adjoint_greeks(S0,K,T,r,sigma,'C')
    t1=time.perf_counter()
    print(f"Vega={v_adj:.6f}, Rho={r_adj:.6f} (time {t1-t0:.4f}s)")

=== BSM Price ===
Price=10.450584 (time 0.0004s)

=== PDE Price (CN scheme) ===
Price=10.440692 (time 0.0909s)

=== BSM Greeks ===
Vega=37.524035, Rho=53.232482 (time 0.0003s)

=== FD Greeks (bumping) ===
Vega=37.578451, Rho=53.232693 (time 0.2726s)

=== Adjoint Greeks (discrete) ===
Vega=37.431199, Rho=53.093309 (time 0.3730s)
